# EDA

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Importamos el dataframe
file_path = Path("content/P1") / "bank_10.pkl"
df = pd.read_pickle(file_path)

#print(df.head(10))

# Vemos el número de variables e instancias
n_instancias, n_variables = df.shape
print(f"{n_instancias} instancias y {n_variables} variables.")

# Agrupamos los tipos de variables (a priori no podemos saber cuales son categoricas u ordinales)
lista_variables = list(df.columns)
print(lista_variables)
var_categoricas_y_ordinales = []
var_numericas = []
for var in lista_variables:
  if df[var].dtype == 'int64':
    var_numericas.append(var)
  else:
    var_categoricas_y_ordinales.append(var)

print(f'{var_numericas} variables numéricas.')
print(f'{var_categoricas_y_ordinales} variables categoricas y ordinales.') # Después de imprimir el df entero, vemos que education es ordinal

# Vemos cardinalidad variables categóricas sin contar NaNs
var_categoricas = [v for v in var_categoricas_y_ordinales if v != "education"]
for var in var_categoricas_y_ordinales:
    print(var, df[var].nunique()) # Vemos que tanto job como month tienen alta cardinalidad (>10 valores únicos)



11000 instancias y 17 variables.
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'deposit']
['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'] variables numéricas.
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'deposit'] variables categoricas y ordinales.
job 12
marital 3
education 4
default 2
housing 2
loan 2
contact 3
month 12
poutcome 4
deposit 2


## Missing values

In [3]:
# Vamos a mirar los valores de las variables categóricas buscando posibles celdas con valores
# missing distintos de None o Null en este caso encontramos 'unknown'
for col in lista_variables:
    print(col, df[col].unique())

# Nótese que job tiene los 2 valores None y 'unknown' 

print("\n---\n")

# Vamos a ver cuantos valores hay missing o unknown de cada feature y que porcentaje representan
n = len(df)
for col in lista_variables:
    n_nan = df[col].isna().sum()
    n_unknown = (df[col] == 'unknown').sum()
    
    if n_nan > 0:
        print(col, "NaN:", n_nan, f"({n_nan/n:.2%})")   
    if n_unknown > 0:
        print(col, "unknown:", n_unknown, f"({n_unknown/n:.2%})")

#Pdays al ser un caso concreto lo veremos ahora
n_minus = (df['pdays']==-1).sum()
print("pdays -1: ", n_minus, f"({n_minus/n:0.2%})")


age [59 56 41 55 54 42 60 37 28 38 30 29 46 31 35 32 49 43 26 40 33 23 48 45
 36 52 53 39 57 51 44 24 50 27 34 47 25 58 61 68 75 22 69 66 85 72 90 67
 71 21 74 65 62 83 70 76 77 19 73 63 20 78 95 64 79 82 18 86 84 87 92 81
 80 93 88 89]
job ['admin.' None 'technician' 'management' 'retired' 'services'
 'blue-collar' 'unemployed' 'entrepreneur' 'housemaid' 'unknown'
 'self-employed' 'student']
marital ['married' None 'single' 'divorced']
education ['secondary' 'tertiary' 'primary' 'unknown']
default ['no' 'yes']
balance [2343   45 1270 ... 1594 2593 -134]
housing ['yes' 'no']
loan ['no' 'yes']
contact ['unknown' 'cellular' 'telephone']
day [ 5  6  7  8  9 12 13 14 15 16 19 20 21 23 26 27 28 29 30  2  3  4 11 17
 18 24  1 10 22 25 31]
month ['may' 'jun' 'jul' 'aug' 'oct' 'nov' 'dec' 'jan' 'feb' 'mar' 'apr' 'sep']
duration [1042 1467 1389 ... 1504  818 1812]
campaign [ 1  2  3  4  6  5  8 11  9 10 15 12 14  7 13 17 29 21 20 16 32 19 25 24
 22 43 18 41 63 27 30 26 23 28 33 31]
pdays [ -1 1

Podemos ver que hay bastantes features que toman valor 'unknow' parece razonable pensar este valor como missing value.

In [ ]:
# Columnas constantes

for col in lista_variables:
    # Obtenemos las frecuencias de todos los valores (incluyendo NaNs)
    counts = df[col].value_counts(dropna=False, normalize=True)
    top_freq = counts.iloc[0]
    nunique = len(counts)

    print(f"{col:<12} | Únicos: {nunique:<5} | Frecuencia mayoritaria: {top_freq:>7.2%} | Valor mayoritario: {counts.index[0]:<10}")

age          | Únicos: 76    | Frecuencia mayoritaria:   4.46% | Valor mayoritario: 31        
job          | Únicos: 13    | Frecuencia mayoritaria:  22.28% | Valor mayoritario: management
marital      | Únicos: 4     | Frecuencia mayoritaria:  56.04% | Valor mayoritario: married   
education    | Únicos: 4     | Frecuencia mayoritaria:  49.03% | Valor mayoritario: secondary 
default      | Únicos: 2     | Frecuencia mayoritaria:  98.48% | Valor mayoritario: no        
balance      | Únicos: 3783  | Frecuencia mayoritaria:   6.88% | Valor mayoritario: 0         
housing      | Únicos: 2     | Frecuencia mayoritaria:  52.74% | Valor mayoritario: no        
loan         | Únicos: 2     | Frecuencia mayoritaria:  86.99% | Valor mayoritario: no        
contact      | Únicos: 3     | Frecuencia mayoritaria:  72.02% | Valor mayoritario: cellular  
day          | Únicos: 31    | Frecuencia mayoritaria:   5.11% | Valor mayoritario: 20        
month        | Únicos: 12    | Frecuencia mayorita

Vemos que la feature default es casi constante con un 98.48% de valores 'no'

También podemos ver que no hay columnas ID.

Este problema se trata de un problema de clasificación por lo que es muy importante ver si está desbalanceado.
En la salida anterior se puede ver que deposit no está desbalanceada poque tiene un 52.55% de valores 'no' y un 47.45% de valores 'si'.